In [ ]:
#importing modules

import numpy as np
import matplotlib.pyplot as plt
from GWFish.modules.fft import fft_lal_timeseries
from GWFish import detection
from scipy.interpolate import interp1d
from scipy.integrate import simpson
import math
import lal
from lal import REAL8TimeSeries, CreateREAL8TimeSeries, REAL8Vector, CreateREAL8Vector
from pathlib import Path

In [ ]:
GWFish_path = Path("..")

filename = "23_gwstrain_trim.dat"
ET_psd=  GWFish_path / "GWFish/detector_psd/ET_psd.txt"

GW_dat = np.loadtxt(filename).T
Et_dat = np.loadtxt( ET_psd ).T

In [ ]:
#setting parameters
geo_time = np.random.uniform(1672531200, 1704067200)
distance = 10 #kpc
kpc = 3.086e21 #cm
dims = 300_000
z=np.random.uniform(1e-20, 0.007)

#change to dataframe
params = {
    #"mass": np.array(14),
    #"redshift":z,
    "ra" : np.random.uniform(0., 2 * np.pi),
    "dec" : np.arcsin(np.random.uniform(-1., 1.)),
    "psi" : np.random.uniform(0., np.pi),
    "max_frequency_cutoff" : 2048,
    "geocent_time": geo_time,
    "theta_j" : np.arccos(np.random.uniform(-1., 1.))
                                       }
#aggiungere l'energia emessa
#cercare un SNR tipico per CCSNE

In [ ]:
#SNR

def make_fft_from_time_series(time_series_input, df, dt, title="Ines_Ludo"):
    '''
    Returns the FFT done through the lal library given a time series. Also returns the frequency range.
    time_series_input: array of the time series 
    df: frequency step
    dt: time step
    title: title of the time series (optional)
    '''
    dims = len(time_series_input)
    time_series = CreateREAL8Vector(dims)
    time_series.data = time_series_input
    ts = CreateREAL8TimeSeries(title, 1, 0, dt, lal.DimensionlessUnit, dims)
    ts.data = time_series
    fft_dat = fft_lal_timeseries(ts, df).data.data
    freq_range = np.linspace( 0, df * len(fft_dat), len(fft_dat) )
    return fft_dat, freq_range

def frequency_plot_options(ax, fig, y_bounds = [1e-25, 1e-22], x_bounds = [1, 1e4]):
    ax.set_yscale("log")
    ax.set_ylabel(f"$|\\tilde{{h}}_{{+/x}}|$")
    ax.set_xlabel("f[Hz]")
    ax.set_xscale("log")
    ax.set_ylim(y_bounds)
    ax.set_xlim(x_bounds)
    ax.grid()
    ax.legend()
    fig.tight_layout()
    return 0

new_time = np.linspace(min(GW_dat[0]), max(GW_dat[0]), dims)
interpolated_data = interp1d(GW_dat[0], GW_dat[1:], axis=1)(new_time)
GW_dat_interp = interpolated_data / (distance * kpc)

fft_h_plus = np.fft.fft(GW_dat_interp[0], dims, norm="forward" ) 
fft_h_cross = np.fft.fft(GW_dat_interp[1], dims, norm="forward" )
frequencies = np.fft.fftfreq(dims, d=np.mean(np.diff(new_time)))


df = frequencies[1] - frequencies[0]
dt = new_time[1] - new_time[0]

fft_dat_plus, freq_range = make_fft_from_time_series(GW_dat_interp[0], df, dt)
fft_dat_cross, _ = make_fft_from_time_series(GW_dat_interp[1], df, dt)

time_dim = dims//2+1
timevector = np.ones( time_dim ) * geo_time
ConfigDet = 'detectors.yaml'
detector = detection.Detector("ET", config=ConfigDet)


phi_in = np.exp(1.j*(2*detector.frequencyvector*np.pi*geo_time)).T[0] #TODO shape is (dims, 1) makes it too high dimensional
fft_dat_plus = phi_in*np.conjugate( fft_dat_plus )
fft_dat_cross = phi_in*np.conjugate( fft_dat_cross )

# GW Fish format for hfp and hfc
hfp = fft_dat_plus[:, np.newaxis]
hfc = fft_dat_cross[:, np.newaxis]
polarizations = np.hstack((hfp, hfc))

args = (params, detector, polarizations, timevector)

signal = detection.projection(*args)
frequencyvector = detector.frequencyvector

frequencyvector = freq_range

component_SNRs = detection.SNR(detector, signal, frequencyvector=np.squeeze(frequencyvector))
out_SNR = np.sqrt(np.sum(component_SNRs**2))
print("SNR:",out_SNR)
print(params)


In [ ]:
from scipy.optimize import brentq
from astropy.cosmology import Planck18
from scipy.integrate import quad

# Setting constraints
SNR_target = 30 
z_min = 0.0005
z_max = 0.007 #
energy_emitted = np.random.uniform(5.325276e40, 5.325276e41)
H0 = 70.0  # Hubble constant in km/s/Mpc
omega_m = 0.3  # Matter density parameter
omega_lambda = 0.7  # Dark energy density parameter
c = 299792.458  # Speed of light in km/s

new_freq = np.linspace(min(Et_dat[0]), max(Et_dat[0]), dims)
psd= interp1d(Et_dat[0], Et_dat[1])(new_freq)

def luminosity_distance(z, H0, omega_m, omega_lambda, c):
    integrand = lambda z: 1 / np.sqrt(omega_m * (1 + z)**3 + omega_lambda)
    integral, _ = quad(integrand, 0, z)
    return (1 + z) * c / H0 * integral

def compute_SNR(z, luminosity_distance, psd_value):
    psd_value = #some function that extrapolate a value of the psd(z)
    SNR = luminosity_distance(z) / np.sqrt(psd_value)
    return SNR, psd_value

def SNR_error(z, psd, SNR_target,luminosity_distance):
    SNR = compute_SNR(z, luminosity_distance, psd_value)
    return np.log(SNR / SNR_target)


# Horizon computation
redshift = brentq(SNR_error, z_min, z_max, args=(psd_value,  SNR_target, luminosity_distance))
print("Redshift:", redshift)
distance = Planck18.luminosity_distance(redshift).value
print("Luminosity distance:", distance)
SNR = compute_SNR(z=redshift, psd_value=psd_value, energy_emitted=energy_emitted)
print("SNR:", SNR)
